In [232]:
# %pip install pyyaml


In [2]:
import pandas as pd
import numpy as np
import math
import random
from dataclasses import dataclass, asdict
from datetime import datetime
from zoneinfo import ZoneInfo
import yaml
from pathlib import Path

In [4]:
# load in df with all bracket info
bracket_df = pd.read_csv("../input_data/final_bracket_df_w_seed.csv").iloc[:, 1:].copy()
bracket_df = bracket_df.loc[bracket_df["season"]==2025]

# load in df with all win prob combinations
win_prob_df = pd.read_csv("../input_data/final_pairs_final.csv").iloc[:, 1:].copy()
win_prob_df = win_prob_df.loc[win_prob_df["season"]==2025]

# load in df with days and round info
dates_df = pd.read_csv("../input_data/date_final_all.csv").iloc[:, 1:].copy()
dates_df = dates_df.loc[dates_df["season"]==2025]

In [235]:
# pull out first four teams
bracket_df_ff = bracket_df.loc[bracket_df["round"]=="First Four",["team1_name", "next_game_id1"]]
bracket_df_ff = bracket_df_ff.rename(columns = {"team1_name":"new_team", "next_game_id1":"next_game_id0"})
bracket_df_ff["join_term"] = "Winner"
ff_teams = bracket_df_ff["new_team"].unique()

# bring in potential ff winners to r1 games
bracket_join = bracket_df.copy()
bracket_join["join_term"] = bracket_join["team1_name"].str[:6]
bracket_df2 = bracket_join.merge(bracket_df_ff, on = ["next_game_id0","join_term"], how = "left")

mask = (bracket_df2["join_term"]=="Winner")
bracket_df2.loc[mask, "team1_name"] = bracket_df2.loc[mask, "new_team"]
bracket_df2 = bracket_df2.drop(["join_term","new_team"], axis=1)
bracket_df2 = bracket_df2.loc[bracket_df2["round"]=="Round of 64"]

# sort teams
bracket_df2 = bracket_df2.sort_values("team1_name")

team_seed_arr_orig = bracket_df2["team1_seed"].to_numpy(dtype = int)

bracket_df2 = bracket_df2.drop(columns=["team1_seed"], axis = 1)


In [236]:
# teams indices in alphabetical order
teams = bracket_df2["team1_name"].sort_values().unique()

In [237]:
# pull out first four matchup pairs of team indices
ff_indices_df = bracket_df2["team1_name"].isin(ff_teams).reset_index()
ff_indices = ff_indices_df[ff_indices_df["team1_name"]==True].index.to_list()

bracket_df_ff_short = bracket_df_ff.sort_values("new_team")
bracket_df_ff_short["id"] = ff_indices

ff_game_ids = list(bracket_df_ff_short["next_game_id0"].unique())

ff_pair_list = []
for gid in ff_game_ids:
    pair = list(bracket_df_ff_short.loc[bracket_df_ff_short["next_game_id0"]==gid, "id"].unique())
    ff_pair_list.append(pair)


In [238]:
# initialize unique game ids for each round
unique_games_r0 = bracket_df2["next_game_id0"].unique().tolist()
unique_games_r1 = bracket_df2["next_game_id1"].unique().tolist()
unique_games_r2 = bracket_df2["next_game_id2"].unique().tolist()
unique_games_r3 = bracket_df2["next_game_id3"].unique().tolist()
unique_games_r4 = bracket_df2["next_game_id4"].unique().tolist()
unique_games_r5 = bracket_df2["next_game_id5"].unique().tolist()


# map alphabetical-ordered IDs to numeric indices for matchups
unique_game_ids = unique_games_r0 + unique_games_r1 + unique_games_r2 + unique_games_r3 + unique_games_r4 + unique_games_r5
unique_game_ids.sort()

unique_game_nums = list(range(0,len(unique_game_ids)))
id_map = dict(zip(unique_game_ids, unique_game_nums))

new_bracket_df = bracket_df2.replace(id_map).sort_values("team1_name").drop(columns=["round", "team1_name", "season"])

unique_games_r0 = [id_map.get(x, x) for x in unique_games_r0]
unique_games_r1 = [id_map.get(x, x) for x in unique_games_r1]
unique_games_r2 = [id_map.get(x, x) for x in unique_games_r2]
unique_games_r3 = [id_map.get(x, x) for x in unique_games_r3]
unique_games_r4 = [id_map.get(x, x) for x in unique_games_r4]
unique_games_r5 = [id_map.get(x, x) for x in unique_games_r5]


# change unique game lists to list of lists structure
unique_games_all = [unique_games_r0, unique_games_r1, unique_games_r2, unique_games_r3, unique_games_r4, unique_games_r5]

# change team, round df to array
bracket_arr_orig = new_bracket_df.to_numpy(dtype=float)

C:\Users\Owner\AppData\Local\Temp\ipykernel_20112\2552703120.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  new_bracket_df = bracket_df2.replace(id_map).sort_values("team1_name").drop(columns=["round", "team1_name", "season"])


In [239]:
# create list of round days that corresponds with unique game_id indices
dates_df_short = dates_df[["game_id", "day_num"]].sort_values("game_id")
dates_arr = dates_df_short["day_num"].to_numpy(dtype=int)

In [240]:
game_day_map = dict(zip(dates_df_short["game_id"], dates_df_short["day_num"]))

bracket_day_arr_orig = bracket_df2.replace(game_day_map).sort_values("team1_name").drop(columns=["round", "team1_name", "season"]).to_numpy()

team_day_list = []
for c in range(3):

    cur_list = bracket_day_arr_orig[:, c]

    cur_list_d1 = (cur_list == 1).astype(int)
    cur_list_d2 = (cur_list == 2).astype(int)

    cur_list_full= [cur_list_d1, cur_list_d2]


    team_day_list.append(cur_list_full)



C:\Users\Owner\AppData\Local\Temp\ipykernel_20112\1026973982.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bracket_day_arr_orig = bracket_df2.replace(game_day_map).sort_values("team1_name").drop(columns=["round", "team1_name", "season"]).to_numpy()


In [241]:
potential_opps_arr_start = []
for r in range(6):
    cut = bracket_arr_orig[:, r]
    potential_opps_arr_cur = (cut[:, None] == cut[None, :]).astype(np.int8)
    np.fill_diagonal(potential_opps_arr_cur, 0)

    potential_opps_arr_start.append(potential_opps_arr_cur)


potential_opps_arr_orig = [potential_opps_arr_start[0].copy()]

for i in range(1, len(potential_opps_arr_start)):
    if i ==1:
        potential_opps_arr_orig.append(potential_opps_arr_start[i] - potential_opps_arr_orig[i-1])
    else:
        potential_opps_arr_orig.append(potential_opps_arr_start[i] - potential_opps_arr_start[i-1])

In [242]:
# create a team, game_id df for which teams play in certain games
id_cols = [c for c in bracket_df2.columns if c.startswith("next_game_id")]
base_cols = ["team1_name"]

long = (
    bracket_df2[base_cols + id_cols]
      .melt(id_vars=base_cols, value_vars=id_cols, value_name="game_id")
      .dropna(subset=["game_id"])
)

team_game_matrix = (
    pd.crosstab(long["team1_name"], long["game_id"])
      .astype(int)
).sort_values("team1_name")

# change to array
game_match_arr_orig = team_game_matrix.to_numpy()


In [243]:
# create new game_id structure so we know differences between day 1 and day 2 games for simulation
unique_games_d1 = []
unique_games_d2 = []

for r in list(range(6)):

    if r <= 2:
    
        all_day1 = np.where(dates_arr==1)
        this_round = np.array(unique_games_all[r])

        mask = np.isin(this_round, all_day1)
        this_round_d1 = this_round[mask]
        this_round_d2 = this_round[~mask]

        unique_games_d1.append(this_round_d1)
        unique_games_d2.append(this_round_d2)
    
    else:
        
        this_round = np.array(unique_games_all[r])
        unique_games_d1.append(this_round)

unique_games_comb = [unique_games_d1, unique_games_d2]

In [244]:
# list for how many days per round
days_by_round = [2,2,2,1,1,1,]

In [245]:
# create array for team, team win probabilities
df = win_prob_df.drop(columns=["season"], errors="ignore").sort_values("cur_team")

idx_cols = [c for c in df.columns if c not in ["opponent", "win_prob"]]

win_prob_wide = (
    df.pivot_table(
        index=idx_cols,
        columns="opponent",
        values="win_prob",
        aggfunc="first"
    )
    .reset_index()
).sort_values("cur_team").sort_index(axis=1).drop("cur_team", axis=1)


wp_arr_orig = np.nan_to_num(win_prob_wide.to_numpy(dtype=float))

In [246]:
# initialize 1D array for teams being still in or not
still_alive_arr_orig = np.ones((68,), dtype=int)


In [247]:

vals_ff = np.unique(bracket_arr_orig[:, 4])
groups_ff_orig = {f"group{i+1}": np.where(bracket_arr_orig[:, 4] == v)[0].tolist()
          for i, v in enumerate(vals_ff)}



vals_ee = np.unique(bracket_arr_orig[:, 3])
groups_ee_orig = {f"group{i+1}": np.where(bracket_arr_orig[:, 3] == v)[0].tolist()
          for i, v in enumerate(vals_ee)}

In [248]:
groups_ee_orig

{'group1': [0, 1, 3, 4, 7, 8, 15, 26, 36, 38, 39, 45, 47, 49, 62, 63, 64],
 'group2': [10,
  17,
  18,
  20,
  21,
  22,
  25,
  31,
  46,
  48,
  53,
  54,
  57,
  59,
  61,
  65,
  66],
 'group3': [2,
  6,
  9,
  13,
  23,
  27,
  28,
  29,
  33,
  34,
  35,
  41,
  43,
  50,
  51,
  55,
  58,
  67],
 'group4': [5, 11, 12, 14, 16, 19, 24, 30, 32, 37, 40, 42, 44, 52, 56, 60]}

## Break

In [249]:
still_alive_arr_orig.shape

(68,)

In [250]:
bracket_arr_orig

array([[ 9.,  3., 14.,  1., 16.,  0.],
       [13.,  5., 15.,  1., 16.,  0.],
       [38., 34., 46., 33., 16.,  0.],
       [ 6.,  2., 14.,  1., 16.,  0.],
       [ 9.,  3., 14.,  1., 16.,  0.],
       [59., 52., 62., 48., 17.,  0.],
       [38., 34., 46., 33., 16.,  0.],
       [10.,  4., 15.,  1., 16.,  0.],
       [ 7.,  2., 14.,  1., 16.,  0.],
       [45., 37., 47., 33., 16.,  0.],
       [25., 20., 31., 18., 17.,  0.],
       [55., 50., 61., 48., 17.,  0.],
       [54., 49., 61., 48., 17.,  0.],
       [39., 34., 46., 33., 16.,  0.],
       [57., 51., 62., 48., 17.,  0.],
       [ 6.,  2., 14.,  1., 16.,  0.],
       [53., 49., 61., 48., 17.,  0.],
       [24., 19., 31., 18., 17.,  0.],
       [24., 19., 31., 18., 17.,  0.],
       [56., 50., 61., 48., 17.,  0.],
       [26., 20., 31., 18., 17.,  0.],
       [23., 19., 31., 18., 17.,  0.],
       [27., 21., 32., 18., 17.,  0.],
       [43., 36., 47., 33., 16.,  0.],
       [59., 52., 62., 48., 17.,  0.],
       [28., 21., 32., 18

In [251]:
game_match_arr_orig

array([[1, 1, 0, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0]], shape=(68, 63))

In [252]:
dates_arr

array([1, 1, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2,
       1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 1, 2, 2, 1, 1, 1, 1, 2, 2,
       2, 2, 2, 2, 1, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1])

In [253]:
wp_arr_orig

array([[0.        , 0.07466937, 0.8470308 , ..., 0.62924956, 0.21865896,
        0.41046466],
       [0.92533063, 0.        , 0.98563628, ..., 0.95461298, 0.77618701,
        0.89613846],
       [0.1529692 , 0.01436372, 0.        , ..., 0.23460269, 0.04810819,
        0.11169479],
       ...,
       [0.37075044, 0.04538702, 0.76539731, ..., 0.        , 0.14154735,
        0.29089451],
       [0.78134104, 0.22381299, 0.95189181, ..., 0.85845265, 0.        ,
        0.71329753],
       [0.58953534, 0.10386154, 0.88830521, ..., 0.70910549, 0.28670247,
        0.        ]], shape=(68, 68))

In [254]:
potential_opps_arr_orig[0]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(68, 68), dtype=int8)

## Break

In [255]:
unique_games_comb[0][0]

array([38, 59, 10, 25, 39, 57, 24, 26, 23, 40, 11, 60, 30, 41, 58, 29])

In [256]:
#game_match_arr = game_match_arr_orig.copy()

np.flatnonzero(game_match_arr[:, 38] == 1)

NameError: name 'game_match_arr' is not defined

In [ ]:
game_match_arr[2, :] = 0

In [ ]:
game_match_arr[2, :]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:

groups = {f"group_{int(v)}": np.where(bracket_arr_orig[:,3] == v)[0].tolist() for v in np.unique(bracket_arr_orig[:,3])}

mapping = {}
for g3 in np.unique(bracket_arr_orig[:,3]):
    r4_vals = np.unique(bracket_arr_orig[:,4][bracket_arr_orig[:,3] == g3])
    if len(r4_vals) != 1:
        raise ValueError(f"Round3 id {g3} maps to multiple round4 ids: {r4_vals}")
    mapping[g3] = r4_vals[0]


In [ ]:


groups_ff_orig

{'group_16': [0,
  1,
  2,
  3,
  4,
  6,
  7,
  8,
  9,
  13,
  15,
  23,
  26,
  27,
  28,
  29,
  33,
  34,
  35,
  36,
  38,
  39,
  41,
  43,
  45,
  47,
  49,
  50,
  51,
  55,
  58,
  62,
  63,
  64,
  67],
 'group_17': [5,
  10,
  11,
  12,
  14,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  24,
  25,
  30,
  31,
  32,
  37,
  40,
  42,
  44,
  46,
  48,
  52,
  53,
  54,
  56,
  57,
  59,
  60,
  61,
  65,
  66]}

In [263]:
@dataclass
class SimConfig:

    survivor_policies : list
    
    n_iter: int = 10

    store_tourney_results: bool = True

    run_survivor : bool = True

    store_picked_teams : bool = True

    wp_arr_orig = wp_arr_orig.copy()
    bracket_arr_orig = bracket_arr_orig.copy()
    game_match_arr_orig = game_match_arr_orig.copy()
    still_alive_arr_orig = still_alive_arr_orig.copy()
    potential_opps_arr_orig = potential_opps_arr_orig.copy()
    team_seed_arr_orig = team_seed_arr_orig.copy()

    groups_ff_orig = groups_ff_orig.copy()
    groups_ee_orig = groups_ee_orig.copy()

    




# --------------------------------------------
# 2) Simulator object (holds state + methods)
# --------------------------------------------
class TournamentSimulator:
    def __init__(self, cfg: SimConfig, n_teams: int = 68):
        
        self.cfg = cfg

        self.n_teams = n_teams

        self.sim_id = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H.%M.%S_%f")

        # Initialize storage for results if requested
        if self.cfg.store_tourney_results:
            self.tourney_results = np.full(
                (self.n_teams, self.cfg.n_iter),
                7,
                dtype=int
            )
        else:
            self.tourney_results = None

        if self.cfg.store_picked_teams:
            chosen_teams_all_save = []

    def clean_team_list(self, x):
        if x is None:
            return ""
        # if x is a list/array of ids (may include None)
        vals = [int(v) for v in np.asarray(x).ravel().tolist() if v is not None]
        return ",".join(map(str, vals))

    def record_result(self, team_id: int, round_: int, iter_: int):
        if self.cfg.store_tourney_results:
            self.tourney_results[team_id, iter_] = round_

    
    def survivor_choose_team(self, interm_wp_arr, potential_opps_arr, team_seed_arr, r_cur, policies_alive, chosen_teams_prev, not_picked, d, still_alive_arr, i, groups_ff, groups_ee):

        opps_by_nr = [1, 2, 4, 8, 16, 32]

        it=-1
        all_rd_wps = []
        for r in range(r_cur,6):

            it+=1
            #print(it)
            #print(r)

            next_wp_arr = potential_opps_arr[r] * interm_wp_arr
            #pd.DataFrame(next_wp_arr).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY1.csv")

            #

            

            #

            if it > 0:
                all_rd_wps_arr_cp_cur = all_rd_wps_arr_cp[:,it-1]
                #pd.DataFrame(all_rd_wps_arr_cp_cur).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY2.csv")
                final_wp_arr = next_wp_arr * all_rd_wps_arr_cp_cur[None, :]
                #final_wp_arr = next_wp_arr * all_rd_wps_arr_cp_cur[:, None]
            else:
                final_wp_arr = next_wp_arr.copy()


            #pd.DataFrame(final_wp_arr).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY3.csv")

            
            #pd.DataFrame(potential_opps_arr[r]).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY1.csv")
            
            # if r == 0:
            #     print(potential_opps_arr[r])
            #     print(interm_wp_arr)
            #     print(final_wp_arr)

            row_sums = final_wp_arr.sum(axis=1)
            #row_nnz  = (final_wp_arr != 0).sum(axis=1)

            #rd_wps = np.divide(row_sums, row_nnz, out=np.zeros_like(row_sums, dtype=float), where=row_nnz != 0)




            #pd.DataFrame(rd_wps).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY1.csv")
            #print(opps_by_nr[it])

            
            

            
            
            #print(rd_wps[1])

            all_rd_wps.append(row_sums)
            all_rd_wps_arr = np.column_stack(all_rd_wps)

            all_rd_wps_arr_cp = np.cumprod(all_rd_wps_arr, axis=1)

        chosen_teams = []
        for p_iter in range(len(policies_alive)):

            if policies_alive[p_iter] == 0:

                if r_cur !=3:
                    chosen_teams.append(None)
                    
                else:
                    chosen_teams.append([None, None])
                
                continue
            
            policy = self.cfg.survivor_policies[p_iter]
            


                #print(all_rd_wps_arr_cp_sum)

            #print(all_rd_wps[0][4])

            # if d==0:
            #     pd.DataFrame(all_rd_wps_arr_cp).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY1.csv")
            # else:
            #     pd.DataFrame(all_rd_wps_arr_cp).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY2.csv")
            #all_rd_adv_probs = np.cumprod(all_rd_wps_arr, axis=0)
            #pd.DataFrame(all_rd_adv_probs).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY3.csv")

            #print(all_rd_adv_probs)

            if r_cur < 5:
                weights_list = np.array(policy["next_round_weights"][f"r{r_cur}"])
                obj_arr = (all_rd_wps_arr * weights_list.T).sum(axis=1)
            else:
                obj_arr = all_rd_wps_arr.sum(axis=1)

            
            available_arr = still_alive_arr * not_picked[p_iter]





            

            
            obj_arr = obj_arr * not_picked[p_iter]


            if r_cur <= 2:
                obj_arr = obj_arr * team_day_list[r_cur][d]




            if ((r_cur in [0,1]) and (policy["ignore_seeds"][f"r{r_cur}"] is not False)):


                keep_team_seed = (~np.isin(team_seed_arr, policy["ignore_seeds"][f"r{r_cur}"])).astype(int).ravel()

                #print(keep_team_seed)

                print(obj_arr)
                print(keep_team_seed)

                # print(obj_arr.shape)
                # print(team_seed_arr.shape)
                # print(keep_team_seed.shape)

                obj_arr = obj_arr * keep_team_seed


                print(obj_arr)



            

            #pd.DataFrame(obj_arr).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY1.csv")
            #print(obj_arr)
            obj_vals = np.sort(obj_arr)[::-1] 
            #print(vals[:5])
            
            team_idx_order = np.argsort(obj_arr)[::-1]
            #print(obj_vals!=0)








            team_idx_order = team_idx_order[obj_vals!=0]


            if (policy["min_region"] is not False) & (r_cur < 3):
                
                region_filter = []
                for g_num in [1, 2, 3, 4]:
                    if np.sum(available_arr[groups_ee_orig[f"group{g_num}"]]) <= policy["min_region"][f"r{r_cur}"]:
                        region_filter.extend(groups_ee_orig[f"group{g_num}"])
                
                
                if len(region_filter)>0:
                    # print(f"Round {r_cur} -- Day {d}")
                    # print(region_filter)
                    # print(" ")

                    # print(team_idx_order)
                    mask_outside = ~np.isin(team_idx_order, region_filter)
                    if mask_outside.any():
                        team_idx_order = team_idx_order[mask_outside]
                        #print(team_idx_order)




            #print(team_idx_order)

            
            #print(team_idx_order[:5])
            #obj_arr_sorted = obj_arr[team_idx_order]

            if team_idx_order.size == 0:
                
                chosen_teams.append(1000)

                #print(f"it {i}")

                continue


            if (policy["smart_last_rounds"] != False) and (r_cur >= 3):

                if r_cur == 3:

                    group_1_idxs = team_idx_order[np.isin(team_idx_order, groups_ff["group1"])]
                    #print(team_idx_order)
                    #print(group_1_idxs)
                    group_2_idxs = team_idx_order[np.isin(team_idx_order, groups_ff["group2"])]

                    if (policy["smart_last_rounds"]["ee_swap"] == True) and (group_1_idxs.size > 1) and (group_2_idxs.size > 1):

                        #print(f"it {i}")


                        chosen_teams.append([int(group_1_idxs[1]), int(group_2_idxs[1])])

                        #print(chosen_teams)

                        not_picked[p_iter][group_1_idxs[1]] = 0
                        not_picked[p_iter][group_2_idxs[1]] = 0

                        continue

                if (r_cur == 4) and (policy["smart_last_rounds"]["ff_swap"] == True) and (team_idx_order.size > 1):

                    chosen_team_cur = team_idx_order[1]
                    chosen_teams.append(chosen_team_cur)
                    not_picked[p_iter][team_idx_order[1]] = 0 

                    continue



            if r_cur != 3:


                if (r_cur in [0, 1]) and (policy["randomness"][f"r{r_cur}"] is not False):

                    random_weights = policy["randomness"][f"r{r_cur}"][:team_idx_order.size + 1]
                    random_weight_sum = sum(random_weights)
                    random_weights = [num / random_weight_sum for num in random_weights]

                    rand_idx = random.choices(range(len(random_weights)), weights=random_weights, k=1)[0]

                    chosen_team_cur = team_idx_order[rand_idx]
                    chosen_teams.append(chosen_team_cur)
                    not_picked[p_iter][team_idx_order[rand_idx]] = 0


                else:

                    chosen_team_cur = team_idx_order[0]
                    chosen_teams.append(chosen_team_cur)
                    not_picked[p_iter][team_idx_order[0]] = 0



                

            else:

                chosen_teams.append([int(team_idx_order[0]), int(team_idx_order[1])])

                not_picked[p_iter][team_idx_order[0]] = 0
                not_picked[p_iter][team_idx_order[1]] = 0

                #print(chosen_team_cur)



        chosen_teams_np = np.array(chosen_teams, dtype=object)



        return chosen_teams_np





            


    def run_sim(self):

        if self.cfg.run_survivor == True:
            policies_round_out_save = []
        if self.cfg.store_picked_teams == True:
            chosen_teams_all_save = []


        for i in range(self.cfg.n_iter):

            print(f"------------ITER: {i}---------")

            wp_arr = self.cfg.wp_arr_orig.copy()
            bracket_arr = self.cfg.bracket_arr_orig.copy()
            game_match_arr = self.cfg.game_match_arr_orig.copy()
            still_alive_arr = self.cfg.still_alive_arr_orig.copy()
            potential_opps_arr = self.cfg.potential_opps_arr_orig.copy()
            team_seed_arr = self.cfg.team_seed_arr_orig.copy()

            groups_ff = groups_ff_orig.copy()
            groups_ee = groups_ee_orig.copy()

            chosen_teams = None

            chosen_teams_all = []

            if self.cfg.run_survivor == True:
                policies_alive = np.ones((len(self.cfg.survivor_policies)), dtype=int)
                not_picked = [np.ones(68, dtype=int).copy() for _ in range(len(self.cfg.survivor_policies))]
                policies_round_out = np.full(len(self.cfg.survivor_policies), 10, dtype=int)


            # sim the first four beforehand
            for g in ff_pair_list:
                t1, t2 = g

                t1_wp = wp_arr[t1][t2]

                rand_num = random.random()

                if rand_num > t1_wp:
                    still_alive_arr[t1] = 0
                    wp_arr[t1, :] = 0
                    wp_arr[:, t1] = 0
                    game_match_arr[t1, :] = 0

                    #print(t2)
                    self.record_result(t1, 0, i)
                else:
                    still_alive_arr[t2] = 0
                    wp_arr[t2, :] = 0
                    wp_arr[:, t2] = 0
                    game_match_arr[t2, :] = 0

                    #print(t1)
                    self.record_result(t2, 0, i)

            # for each round
            day_cum = -1
            for r in list(range(0,6)):

                # for each day of the round
                for d in list(range(0,days_by_round[r])):

                    if r !=3:
                        day_cum += 1
                    else:
                        day_cum += 2

                    if self.cfg.run_survivor == True:
                        if r <=2:
                            #interm_wp_arr = wp_arr * team_day_list[r][d][:, None]
                            interm_wp_arr = wp_arr * 1


                            #pd.DataFrame(team_day_list[r][d][:, None]).to_csv(f"sim_saves/tourney_results/run_{self.sim_id}_ARRAY1.csv")
                        else:
                            interm_wp_arr = wp_arr * 1
                            #print(policies_alive)
                        chosen_teams = self.survivor_choose_team(interm_wp_arr, potential_opps_arr, team_seed_arr, r, policies_alive, chosen_teams, not_picked, d, still_alive_arr, i, groups_ff, groups_ee)

                    # for each game in the round/day
                    for g in unique_games_comb[d][r]:

                        #pull the two teams out of the game
                        t1, t2 = np.flatnonzero(game_match_arr[:, g] == 1)
                        #print([t1,t2])

                        # find the win prob for teams 1
                        t1_wp = wp_arr[t1][t2]
                        #print(t1_wp)

                        # generate number to determine winner
                        rand_num = random.random()
                        #print(rand_num)

                        # update matrices to reflect winner
                        if rand_num > t1_wp:
                            # get rid of team in still alive array
                            still_alive_arr[t1] = 0
                            wp_arr[t1, :] = 0
                            wp_arr[:, t1] = 0
                            # get rid of team's game_ids
                            game_match_arr[t1, :] = 0

                            self.record_result(t1, r+1, i)
                        else:
                            still_alive_arr[t2] = 0
                            wp_arr[t2, :] = 0
                            wp_arr[:, t2] = 0
                            game_match_arr[t2, :] = 0

                            self.record_result(t2, r+1, i)
                    
                    if self.cfg.run_survivor == True:

                        #print(" ")
                        #print(day_cum)


                        if r != 3:
                            #print(chosen_teams)
                            chosen_teams_all.append(chosen_teams)

                            chosen_alive = np.zeros(len(chosen_teams), dtype=int) 
                            mask = (chosen_teams != None) & (chosen_teams != 1000)
                            idx = chosen_teams[mask].astype(int)
                            chosen_alive[mask] = still_alive_arr[idx].astype(int)

                            new_outs = (np.array(policies_alive) + np.array(chosen_alive))


                            policies_alive = np.array(policies_alive) * np.array(chosen_alive)

                            

                            policies_round_out[new_outs == 1] = day_cum


                            policies_alive[chosen_teams == 1000] = 0
                            policies_round_out[chosen_teams == 1000] = day_cum
                        
                        else:
                            
                            #print(chosen_teams)
                            chosen_teams_all.append(chosen_teams[:,0])
                            chosen_teams_all.append(chosen_teams[:,1])


                            chosen_teams0 = chosen_teams[:,0]
                            chosen_alive0 = np.zeros(len(chosen_teams0), dtype=int) 
                            mask0 = chosen_teams0 != None
                            idx0 = chosen_teams0[mask0].astype(int)
                            chosen_alive0[mask0] = still_alive_arr[idx0].astype(int)

                            new_outs0 = (np.array(policies_alive) + np.array(chosen_alive0))

                            #print(new_outs0)



                            chosen_teams1 = chosen_teams[:,1]
                            chosen_alive1 = np.zeros(len(chosen_teams1), dtype=int) 
                            mask1 = chosen_teams1 != None
                            idx1 = chosen_teams1[mask1].astype(int)
                            chosen_alive1[mask1] = still_alive_arr[idx1].astype(int)
                            new_outs1 = (np.array(policies_alive) + np.array(chosen_alive1))

                            #print(new_outs1)


                            policies_alive = np.array(policies_alive) * np.array(chosen_alive0) * np.array(chosen_alive1)

                            


                            new_outs = new_outs0 + new_outs1
                            
                            policies_round_out[new_outs == 3] = day_cum
                            policies_round_out[new_outs == 2] = day_cum-1

                        
                        
                        #print(chosen_teams)
                        # print(policies_round_out)


            if self.cfg.run_survivor == True:
                policies_round_out_save.append(list(policies_round_out))

                
            #print(chosen_teams_all)
            if self.cfg.store_picked_teams == True:
                def to_slot_strings(run):
                    arrs = [np.asarray(x, dtype=object).ravel() for x in run]
                    L = len(arrs[0])
                    out = []
                    for j in range(L):
                        vals = []
                        for a in arrs:
                            v = a[j]
                            vals.append("None" if v is None else str(int(v)))
                        out.append(",".join(vals))
                    return out

                # example usage for ONE run:
                chosen_teams_all_mod = to_slot_strings(chosen_teams_all)
                chosen_teams_all_save.append(chosen_teams_all_mod)
                #print(chosen_teams_all_save)





        #print(policies_round_out_save)


        if (self.cfg.run_survivor == True) or (self.cfg.store_tourney_results == True) or (self.cfg.store_picked_teams == True):
            out_path = Path(f"sim_saves/run_{self.sim_id}")
            out_path.mkdir(parents=True, exist_ok=True)

        
        if self.cfg.run_survivor == True:
            survivor_results_df = pd.DataFrame({i: col for i, col in enumerate(policies_round_out_save)})
            survivor_results_df.to_csv(f"sim_saves/run_{self.sim_id}/run_{self.sim_id}_survivor_results.csv")

            cfg_dict = asdict(cfg)
            with open(f"sim_saves/run_{self.sim_id}/run_{self.sim_id}_config.yaml", "w") as f:
                yaml.safe_dump(cfg_dict, f, sort_keys=False)

        
        if self.cfg.store_picked_teams == True:
            #print(chosen_teams_all_save)
            chosen_team_results_df = pd.DataFrame(chosen_teams_all_save).T
            chosen_team_results_df.to_csv(f"sim_saves/run_{self.sim_id}/run_{self.sim_id}_chosen_teams.csv")


        if self.cfg.store_tourney_results == True:
            tourney_results_pdf = pd.DataFrame(self.tourney_results)
            tourney_results_pdf.to_csv(f"sim_saves/run_{self.sim_id}/run_{self.sim_id}_tourney_results.csv")

        


        

policy_list = [
    {
        "next_round_weights": 
        {
            "r0":[1,0,0,0,0,0,],
            "r1":[1,0,0,0,0,],
            "r2":[1,0,0,0,],
            "r3":[1,0,0,],
            "r4":[1,0],
        },

        "smart_last_rounds" : 
        {
            "ee_swap":True,
            "ff_swap":True
        },
        "randomness":
        {
            "r0": [.5, .3, .1],
            "r1": [.7, .2, .1],
        },
        "min_region":
        {
            "r0": 2,
            "r1": 3,
            "r2": 1
        },
        "ignore_seeds":
        {
            "r0": [1, 2],
            "r1": [1]
        }
    },
    {
        "next_round_weights": 
        {
            "r0":[1,0,0,0,0,0,],
            "r1":[1,0,0,0,0,],
            "r2":[1,0,0,0,],
            "r3":[1,0,0,],
            "r4":[1,0],
        },

        "smart_last_rounds" : 
        {
            "ee_swap":True,
            "ff_swap":True
        },
        "randomness":
        {
            "r0": [.5, .3, .1],
            "r1": [.7, .2, .1],
        },
        "min_region":
        {
            "r0": 2,
            "r1": 3,
            "r2": 1
        },
        "ignore_seeds":
        {
            "r0": False,
            "r1": False
        }
    },
]


# --------------------------------------------
# 3) Example usage
# --------------------------------------------
cfg = SimConfig(n_iter=1, store_tourney_results=True, run_survivor=True, store_picked_teams=True, survivor_policies=policy_list)
sim = TournamentSimulator(cfg, n_teams=68)

sim.run_sim()



------------ITER: 0---------
[0.         0.         0.         0.         0.         0.32776627
 0.88465198 0.62551441 0.         0.         0.76540182 0.
 0.         0.37154252 0.24588108 0.         0.         0.34720339
 0.65279661 0.         0.1687293  0.9891191  0.         0.
 0.67223373 0.         0.         0.         0.62845748 0.
 0.         0.23459818 0.         0.6163319  0.         0.
 0.         0.75411892 0.07806777 0.         0.09122554 0.
 0.         0.         0.         0.         0.8312707  0.
 0.0108809  0.         0.         0.11534802 0.90877446 0.94860122
 0.         0.80476414 0.922266   0.         0.3836681  0.66641717
 0.077734   0.33358283 0.37448559 0.         0.92193223 0.05139878
 0.         0.19523586]
[1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[0.         0.         0.         0.         0.         0.32776627
 0.         0.62551441 0.         0.         0.76540